<a href="https://colab.research.google.com/github/Taimdev9/TRL-AI/blob/main/notebooks/01_first_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("⏳ جاري تحميل الموديل...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ تم تحميل الموديل بنجاح!")

⏳ جاري تحميل الموديل...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ تم تحميل الموديل بنجاح!


In [ ]:
# نجهّز السؤال بصيغة Qwen
messages = [
    {"role": "system", "content": "أنت مساعد ذكي ترد بالعربية."},
    {"role": "user", "content": "مرحبا! عرّفني عن نفسك في سطرين."}
]

# نحول الرسائل لصيغة الموديل
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# نحوّل النص لأرقام
inputs = tokenizer(text, return_tensors="pt").to(model.device)

# نولّد الرد
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True
    )

# نحول الرد من أرقام لنص
response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print("🤖 الرد:")
print(response)

🤖 الرد:
أنا مساعد ذكي مبرمج لتقديم المساعدة وحل المشكلات. يمكنني مساعدة في الكثير من المواضيع، بما في ذلك البحث على الإنترنت، إنشاء المقالات، التوجيه في العديد من الأنشطة اليومية وغيرها.


In [6]:
def chat(question):
    messages = [
        {"role": "system", "content": "أنت مساعد ذكي ترد بالعربية."},
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )
    return response


# نجرب 3 أسئلة
questions = [
    "ما هي عاصمة الأردن؟",
    "اكتب لي جملة ترحيب قصيرة.",
    "ما هو 2 + 2؟"
]

for q in questions:
    print(f"👤 أنت: {q}")
    print(f"🤖 AI: {chat(q)}")
    print("-" * 50)

👤 أنت: ما هي عاصمة الأردن؟
🤖 AI: عاصمة الأردن هي عمان.
--------------------------------------------------
👤 أنت: اكتب لي جملة ترحيب قصيرة.
🤖 AI: مرحبا! كيف يمكنني مساعدتك اليوم؟
--------------------------------------------------
👤 أنت: ما هو 2 + 2؟
🤖 AI: 2 + 2 = 4
--------------------------------------------------


In [7]:
import json
import os
from datetime import datetime

MEMORY_FILE = "memory.json"

def load_memory():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def save_memory(memory):
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(memory, f, ensure_ascii=False, indent=2)

def add_interaction(question, answer, rating=None, correction=None):
    memory = load_memory()
    memory.append({
        "id": len(memory) + 1,
        "question": question,
        "answer": answer,
        "rating": rating,
        "correction": correction,
        "timestamp": datetime.now().isoformat()
    })
    save_memory(memory)

print("✅ نظام الذاكرة جاهز")

✅ نظام الذاكرة جاهز


In [8]:
def chat_with_memory(question):
    answer = chat(question)
    add_interaction(question, answer)
    return answer

# نجرب
print("👤 أنت: ما هي عاصمة سوريا؟")
print("🤖 AI:", chat_with_memory("ما هي عاصمة سوريا؟"))

print("\n👤 أنت: ما هي عاصمة لبنان؟")
print("🤖 AI:", chat_with_memory("ما هي عاصمة لبنان؟"))

👤 أنت: ما هي عاصمة سوريا؟
🤖 AI: عاصمة سوريا هي دمشق.

👤 أنت: ما هي عاصمة لبنان؟
🤖 AI: عاصمة لبنان هي بيروت.


In [9]:
memory = load_memory()
print(f"📊 عدد التفاعلات المحفوظة: {len(memory)}")
print("\nآخر التفاعلات:")
for item in memory:
    print(f"\n#{item['id']}")
    print(f"  ❓ {item['question']}")
    print(f"  💬 {item['answer']}")
    print(f"  ⭐ تقييم: {item['rating']}")
    print(f"  ✏️ تصحيح: {item['correction']}")

📊 عدد التفاعلات المحفوظة: 2

آخر التفاعلات:

#1
  ❓ ما هي عاصمة سوريا؟
  💬 عاصمة سوريا هي دمشق.
  ⭐ تقييم: None
  ✏️ تصحيح: None

#2
  ❓ ما هي عاصمة لبنان؟
  💬 عاصمة لبنان هي بيروت.
  ⭐ تقييم: None
  ✏️ تصحيح: None


In [10]:
def rate_interaction(interaction_id, rating):
    """يحدد تقييم لتفاعل موجود (1 = 👍, -1 = 👎)"""
    memory = load_memory()
    for item in memory:
        if item["id"] == interaction_id:
            item["rating"] = rating
            save_memory(memory)
            return True
    return False

def correct_interaction(interaction_id, correction):
    """يضيف تصحيح للجواب"""
    memory = load_memory()
    for item in memory:
        if item["id"] == interaction_id:
            item["correction"] = correction
            save_memory(memory)
            return True
    return False

def show_last_interactions(n=3):
    """يعرض آخر n تفاعلات مع إمكانية التقييم"""
    memory = load_memory()
    last = memory[-n:]
    for item in last:
        rating_icon = "👍" if item["rating"] == 1 else ("👎" if item["rating"] == -1 else "⚪")
        print(f"#{item['id']} {rating_icon}")
        print(f"  ❓ {item['question']}")
        print(f"  💬 {item['answer']}")
        if item["correction"]:
            print(f"  ✏️ التصحيح: {item['correction']}")
        print("-" * 50)

print("✅ نظام التقييم جاهز")

✅ نظام التقييم جاهز


In [11]:
# نسأل سؤال جديد
print("👤 أنت: ما هي عاصمة مصر؟")
answer = chat_with_memory("ما هي عاصمة مصر؟")
print("🤖 AI:", answer)

# نعرض آخر التفاعلات
print("\n📋 آخر التفاعلات:")
show_last_interactions(5)

👤 أنت: ما هي عاصمة مصر؟
🤖 AI: عاصمة مصر هي القاهرة.

📋 آخر التفاعلات:
#1 ⚪
  ❓ ما هي عاصمة سوريا؟
  💬 عاصمة سوريا هي دمشق.
--------------------------------------------------
#2 ⚪
  ❓ ما هي عاصمة لبنان؟
  💬 عاصمة لبنان هي بيروت.
--------------------------------------------------
#3 ⚪
  ❓ ما هي عاصمة مصر؟
  💬 عاصمة مصر هي القاهرة.
--------------------------------------------------


In [12]:
# نقيّم آخر تفاعل بـ 👍 (1)
memory = load_memory()
last_id = memory[-1]["id"]

rate_interaction(last_id, 1)
print(f"✅ تم تقييم التفاعل #{last_id} بـ 👍")

# نعرض التفاعلات مرة ثانية
print("\n📋 بعد التقييم:")
show_last_interactions(5)

✅ تم تقييم التفاعل #3 بـ 👍

📋 بعد التقييم:
#1 ⚪
  ❓ ما هي عاصمة سوريا؟
  💬 عاصمة سوريا هي دمشق.
--------------------------------------------------
#2 ⚪
  ❓ ما هي عاصمة لبنان؟
  💬 عاصمة لبنان هي بيروت.
--------------------------------------------------
#3 👍
  ❓ ما هي عاصمة مصر؟
  💬 عاصمة مصر هي القاهرة.
--------------------------------------------------


In [13]:
# نسأل سؤال
print("👤 أنت: من مؤسس شركة آبل؟")
answer = chat_with_memory("من مؤسس شركة آبل؟")
print("🤖 AI:", answer)

# نعرض آخر تفاعل
print("\n📋 آخر التفاعل:")
show_last_interactions(1)

👤 أنت: من مؤسس شركة آبل؟
🤖 AI: مؤسس شركة آبل هو ستيف جوبز (Steve Jobs).

📋 آخر التفاعل:
#4 ⚪
  ❓ من مؤسس شركة آبل؟
  💬 مؤسس شركة آبل هو ستيف جوبز (Steve Jobs).
--------------------------------------------------


In [14]:
# نفترض إن آخر تفاعل هو اللي بدنا نصححه
memory = load_memory()
last_id = memory[-1]["id"]

# نضيف التصحيح
rate_interaction(last_id, -1)  # 👎 لأن الجواب غلط
correct_interaction(last_id, "مؤسس شركة آبل هو ستيف جوبز، ستيف وزنياك، ورونالد واين عام 1976.")

print(f"✅ تم تصحيح التفاعل #{last_id}")

# نعرض التفاعلات
print("\n📋 بعد التصحيح:")
show_last_interactions(3)

✅ تم تصحيح التفاعل #4

📋 بعد التصحيح:
#2 ⚪
  ❓ ما هي عاصمة لبنان؟
  💬 عاصمة لبنان هي بيروت.
--------------------------------------------------
#3 👍
  ❓ ما هي عاصمة مصر؟
  💬 عاصمة مصر هي القاهرة.
--------------------------------------------------
#4 👎
  ❓ من مؤسس شركة آبل؟
  💬 مؤسس شركة آبل هو ستيف جوبز (Steve Jobs).
  ✏️ التصحيح: مؤسس شركة آبل هو ستيف جوبز، ستيف وزنياك، ورونالد واين عام 1976.
--------------------------------------------------


In [15]:
# نحفظ الذاكرة كملف عشان نرفعه لاحقًا
with open("memory.json", "r", encoding="utf-8") as f:
    content = f.read()

print("📄 محتوى ملف الذاكرة:")
print(content)
print(f"\n📊 عدد التفاعلات: {len(load_memory())}")

📄 محتوى ملف الذاكرة:
[
  {
    "id": 1,
    "question": "ما هي عاصمة سوريا؟",
    "answer": "عاصمة سوريا هي دمشق.",
    "rating": null,
    "correction": null,
    "timestamp": "2026-10-13T21:54:07.407360"
  },
  {
    "id": 2,
    "question": "ما هي عاصمة لبنان؟",
    "answer": "عاصمة لبنان هي بيروت.",
    "rating": null,
    "correction": null,
    "timestamp": "2026-10-13T21:54:07.876130"
  },
  {
    "id": 3,
    "question": "ما هي عاصمة مصر؟",
    "answer": "عاصمة مصر هي القاهرة.",
    "rating": 1,
    "correction": null,
    "timestamp": "2026-10-13T21:54:47.609758"
  },
  {
    "id": 4,
    "question": "من مؤسس شركة آبل؟",
    "answer": "مؤسس شركة آبل هو ستيف جوبز (Steve Jobs).",
    "rating": -1,
    "correction": "مؤسس شركة آبل هو ستيف جوبز، ستيف وزنياك، ورونالد واين عام 1976.",
    "timestamp": "2026-10-13T21:55:26.687625"
  }
]

📊 عدد التفاعلات: 4
